In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Build features

In [5]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df_num = builder.build_num_features().set_index("id")
anime_genres_df = builder.build_genre_features().set_index("anime_id")
synopsis_tfidf, synopsis_features = builder.build_synopsis_features()
synopsis_svd_df = builder.apply_svd(synopsis_tfidf, synopsis_features.index)

anime_df_complete = pd.concat([
    anime_df_num,
    anime_genres_df,
    synopsis_svd_df,
], axis=1).dropna()

builder.svd_explained_variance

np.float64(0.41006426159563164)

In [25]:
# Pick the feature set to use below.
# anime_df = anime_df_complete
# anime_df = anime_df_num.dropna()
anime_df = anime_genres_df.dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, synopsis_svd_df], axis=1).dropna()

Convert each anime in df to vectors

In [26]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [27]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Bayesion Ridge Regression

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.linear_model import BayesianRidge
import numpy as np

rated_items, anime_df, anime_vectors, anime_df_scaled, builder = anime_data_client.get_rated_items(
    user_scores=user_scores,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
    anime_df=anime_df,
    anime_vectors=anime_vectors,
)

X = np.array([vec for _, vec, _ in rated_items])
y = np.array([score for _, _, score in rated_items], dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=None
)

model = BayesianRidge()
model.fit(X_train, y_train)

pred, pred_std = model.predict(X_test, return_std=True)
pred = np.clip(pred, 1, 10)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)

baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Improvement vs baseline: {baseline_mae - mae:.3f}")

MAE: 1.242
RMSE: 1.479
Baseline MAE: 1.258
Improvement vs baseline: 0.016


In [29]:
print("num ratings:", len(y))
print("mean:", y.mean())
print("median:", np.median(y))
print("std:", y.std())
print(pd.Series(y).value_counts().sort_index())

num ratings: 260
mean: 7.403846153846154
median: 7.0
std: 1.5696596321963387
1.0      1
3.0      2
4.0      8
5.0     10
6.0     57
7.0     57
8.0     48
9.0     59
10.0    18
Name: count, dtype: int64


Hit Rate

In [30]:
# Hide some high-rated anime, train on the rest, and see if the hidden likes show up near the top.
hit_rating_threshold = np.median(y) + 0.5 * np.std(y)
heldout_fraction = 0.25
top_ks = [5, 10, 20, 50, 100]

rated_eval = pd.DataFrame({
    "anime_id": [anime_id for anime_id, _, _ in rated_items],
    "score": [score for _, _, score in rated_items],
})

liked_eval = rated_eval[rated_eval["score"] >= hit_rating_threshold]
if len(liked_eval) < 2:
    raise ValueError("Need at least 2 high-rated anime to run a hit-rate holdout test.")

heldout_liked = liked_eval.sample(frac=heldout_fraction, random_state=None)
heldout_ids = set(heldout_liked["anime_id"])

train_eval = rated_eval[~rated_eval["anime_id"].isin(heldout_ids)]
train_ids = train_eval["anime_id"].tolist()

X_hit_train = anime_df_scaled.loc[train_ids].to_numpy()
y_hit_train = train_eval["score"].to_numpy(dtype=float)

hit_model = BayesianRidge()
hit_model.fit(X_hit_train, y_hit_train)

# Candidates are everything the model did not train on, including the hidden liked anime.
candidate_ids = [anime_id for anime_id in anime_df_scaled.index if anime_id not in set(train_ids)]
X_hit_candidates = anime_df_scaled.loc[candidate_ids].to_numpy()

hit_pred, hit_pred_std = hit_model.predict(X_hit_candidates, return_std=True)
hit_pred = np.clip(hit_pred, 1, 10)

title_by_id = {int(anime["id"]): anime["title"] for anime in anime_data.values()}
hit_recommendations = pd.DataFrame({
    "anime_id": candidate_ids,
    "title": [title_by_id.get(int(anime_id), "Unknown") for anime_id in candidate_ids],
    "actual_score": [user_scores.get(int(anime_id)) for anime_id in candidate_ids],
    "predicted_score": hit_pred,
    "uncertainty_raw": hit_pred_std,
})

uncertainty_clipped = hit_recommendations["uncertainty_raw"].clip(
    lower=hit_recommendations["uncertainty_raw"].quantile(0.05),
    upper=hit_recommendations["uncertainty_raw"].quantile(0.95),
)
if uncertainty_clipped.max() == uncertainty_clipped.min():
    hit_recommendations["uncertainty"] = 0.0
else:
    hit_recommendations["uncertainty"] = (
        (uncertainty_clipped - uncertainty_clipped.min())
        / (uncertainty_clipped.max() - uncertainty_clipped.min())
    )

hit_recommendations["ranking_score"] = (
    hit_recommendations["predicted_score"] - 7.5 * hit_recommendations["uncertainty"]
)
hit_recommendations["is_hidden_like"] = hit_recommendations["anime_id"].isin(heldout_ids)
hit_recommendations = hit_recommendations.sort_values("ranking_score", ascending=False)

hit_rows = []
for k in top_ks:
    top_k = hit_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    hit_rows.append({
        "k": k,
        "hits": hits,
        "heldout_likes": len(heldout_ids),
        "hit_rate": hits / len(heldout_ids),
        "precision_at_k": hits / k,
    })

hit_rate_results = pd.DataFrame(hit_rows)

print(f"Total rated anime: {len(rated_eval)}")
print(f"High-rated anime (score >= {hit_rating_threshold}): {len(liked_eval)}")
print(f"Hidden liked anime: {len(heldout_ids)}")
display(hit_rate_results)

# hit_recommendations[hit_recommendations["is_hidden_like"]].head(20)

Total rated anime: 260
High-rated anime (score >= 7.78482981609817): 125
Hidden liked anime: 31


,k,hits,heldout_likes,hit_rate,precision_at_k
0,5,0,31,0.000000,0.00
1,10,0,31,0.000000,0.00
2,20,0,31,0.000000,0.00
3,50,0,31,0.000000,0.00
4,100,1,31,0.032258,0.01


In [ ]:
baseline_recommendations = hit_recommendations.copy()

# If "mean" is available in anime_df, rank by global MAL mean.
baseline_recommendations["baseline_score"] = anime_df_complete.loc[
    baseline_recommendations["anime_id"], "mean"
].to_numpy()

baseline_recommendations = baseline_recommendations.sort_values(
    "baseline_score",
    ascending=False
)

baseline_rows = []
for k in top_ks:
    top_k = baseline_recommendations.head(k)
    hits = int(top_k["is_hidden_like"].sum())
    baseline_rows.append({
        "k": k,
        "baseline_hits": hits,
        "heldout_likes": len(heldout_ids),
        "baseline_hit_rate": hits / len(heldout_ids),
        "baseline_precision_at_k": hits / k,
    })

baseline_hit_rate_results = pd.DataFrame(baseline_rows)
hit_rate_results.merge(baseline_hit_rate_results, on=["k", "heldout_likes"])

,k,hits,heldout_likes,hit_rate,precision_at_k,baseline_hits,baseline_hit_rate,baseline_precision_at_k
0,5,0,31,0.000000,0.00,0,0.000000,0.00
1,10,0,31,0.000000,0.00,0,0.000000,0.00
2,20,0,31,0.000000,0.00,1,0.032258,0.05
3,50,0,31,0.000000,0.00,3,0.096774,0.06
4,100,1,31,0.032258,0.01,7,0.225806,0.07


Hit Rate

In [32]:
from anime_evaluation import HitRateEvaluator

n_runs = 50
result_top_ks = (5, 10)
uncertainty_weight = 7.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = evaluator.tune_bayesian_uncertainty(
    weights=[uncertainty_weight],
    n_runs=n_runs,
    top_ks=result_top_ks,
    random_state=42,
)

if baseline_summary is not None and not baseline_summary.empty:
    average_metrics = bayesian_summary.merge(
        baseline_summary,
        on="k",
        how="left",
    )
else:
    average_metrics = bayesian_summary.copy()

average_metrics = average_metrics.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics

TypeError: Can only merge Series or DataFrame objects, a <class 'NoneType'> was passed

## Results

The best-performing feature set was **all_features**, which combines numeric metadata, genre one-hot features, and synopsis SVD features. The final notebook check above uses **50 repeated holdout runs** with Bayesian Ridge and an uncertainty weight of **4.5**.

| Feature set | P@5 | P@10 | Improvement over baseline |
| --- | ---: | ---: | ---: |
| all_features | 0.420 | 0.358 | 4.38x / 5.59x |
| numeric_genres | 0.264 | 0.258 | 3.47x / 4.96x |
| numeric_svd | 0.324 | 0.238 | 3.11x / 3.31x |
| only_numeric | 0.172 | 0.146 | 1.72x / 2.70x |
| only_genres | 0.072 | 0.130 | 0.55x / 1.81x |
| genres_svd | 0.128 | 0.098 | 1.39x / 1.81x |

The selected-feature result above was generated over **50 runs** for user `chekkit` from `metrics/feature_set_selection_20260615_153918.csv`, with **253 rated anime**.

Conclusion: keep **all_features** as the selected feature representation for the recommender because it gives the best Precision@5 and Precision@10 while substantially outperforming the global mean baseline.